# Q1 dose-response and threshold sensitivity

**Question:** How do spline and two-piecewise Cox models describe the vitamin D dose-response?

**Inputs:** The prepared Q1 analysis cohort.

**Outputs:** Spline, threshold and two-piecewise Cox summaries and figures.

This retains the existing exploratory threshold search and model specification; it does not turn the selected threshold into a prespecified causal boundary.


## Configuration and cohort


In [ ]:
from pathlib import Path
import os
import sys

if os.environ.get("CAPSTONE_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["CAPSTONE_PROJECT_ROOT"]).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "shared" / "q1q2_common.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the example folder, or set CAPSTONE_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "shared"))
from q1q2_common import *

base = load_q1_analysis_base()


## Dose-response definitions


In [ ]:
def rcs_basis(x: np.ndarray, knots: np.ndarray) -> np.ndarray:
    x = np.asarray(x, float)
    t = np.sort(np.asarray(knots, float))
    k = len(t)
    cube = lambda v: np.where(v > 0, v**3, 0.0)
    den = (t[k - 1] - t[0]) ** 2
    cols = [x]
    for j in range(k - 2):
        tj = t[j]
        cols.append(
            (
                cube(x - tj)
                - cube(x - t[k - 2]) * (t[k - 1] - tj) / (t[k - 1] - t[k - 2])
                + cube(x - t[k - 1]) * (t[k - 2] - tj) / (t[k - 1] - t[k - 2])
            )
            / den
        )
    return np.column_stack(cols)


def run_q1_spline_and_threshold(base: pd.DataFrame) -> None:
    d = base.copy()
    d["vitd"] = pd.to_numeric(d["vitd_main"], errors="coerce")
    d = d.dropna(subset=["vitd", "follow_up_years", "event_mi_or_stroke"] + Q1_COVARS)

    knot_options = {3: [10, 50, 90], 4: [5, 35, 65, 95], 5: [5, 27.5, 50, 72.5, 95]}
    fits = {}
    for k, percentiles in knot_options.items():
        knots = np.percentile(d["vitd"], percentiles)
        basis = rcs_basis(d["vitd"].values, knots)
        spline_cols = [f"s{i}" for i in range(basis.shape[1])]
        m = pd.DataFrame(basis, columns=spline_cols)
        m["follow_up_years"] = d["follow_up_years"].values
        m["event_mi_or_stroke"] = d["event_mi_or_stroke"].values
        for covar in Q1_COVARS:
            m[covar] = d[covar].values
        cph = CoxPHFitter().fit(m, "follow_up_years", "event_mi_or_stroke")
        aic = -2 * cph.log_likelihood_ + 2 * (m.shape[1] - 2)
        fits[k] = {"model": cph, "knots": knots, "cols": spline_cols, "aic": aic, "ll": cph.log_likelihood_}

    best_k = min(fits, key=lambda k: fits[k]["aic"])
    fit = fits[best_k]

    linear = pd.DataFrame(
        {
            "vitd": d["vitd"].values,
            "follow_up_years": d["follow_up_years"].values,
            "event_mi_or_stroke": d["event_mi_or_stroke"].values,
        }
    )
    for covar in Q1_COVARS:
        linear[covar] = d[covar].values
    cph_linear = CoxPHFitter().fit(linear, "follow_up_years", "event_mi_or_stroke")
    lr_nonlinear = 2 * (fit["ll"] - cph_linear.log_likelihood_)
    p_nonlinear = stats.chi2.sf(lr_nonlinear, len(fit["cols"]) - 1)

    beta = fit["model"].params_[fit["cols"]].values
    var = fit["model"].variance_matrix_.loc[fit["cols"], fit["cols"]].values
    reference = 30.0
    grid = np.linspace(np.percentile(d["vitd"], 1), np.percentile(d["vitd"], 99), 300)
    diff = rcs_basis(grid, fit["knots"]) - rcs_basis(np.array([reference]), fit["knots"])
    log_hr = diff @ beta
    se = np.sqrt(np.einsum("ij,jk,ik->i", diff, var, diff))
    hr = np.exp(log_hr)
    lo = np.exp(log_hr - 1.96 * se)
    hi = np.exp(log_hr + 1.96 * se)

    xiao_ng = 27.70 / NMOL_PER_NG
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.fill_between(grid, lo, hi, color=WCM_NAVY, alpha=0.18, label="95% CI")
    ax.plot(grid, hr, color=WCM_RED, lw=2.4, label=f"Adjusted HR ({best_k}-knot RCS)")
    ax.axhline(1.0, color="grey", ls="--", lw=1)
    ax.axvline(reference, color="grey", ls=":", lw=1)
    ax.axvline(xiao_ng, color="green", ls="-.", lw=1.6, label=f"Xiao 2022 = {xiao_ng:.1f} ng/mL")
    ax.set_yscale("log")
    ax.set_xlabel("Baseline serum 25(OH)D, ng/mL", fontsize=12)
    ax.set_ylabel(f"Adjusted hazard ratio (reference = {reference:.0f} ng/mL)", fontsize=12)
    ax.set_title(
        "Dose-response: vitamin D and incident MI or stroke\n"
        f"Restricted cubic spline Cox, n = {len(d):,}, nonlinearity p = {p_nonlinear:.1e}",
        fontsize=11.5,
    )
    ax.legend(fontsize=9.5)
    ax.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    plt.savefig(figure_path("q1_spline_doseresponse.png"), dpi=170, bbox_inches="tight")
    plt.show()
    pd.DataFrame({"vitd": grid, "hr": hr, "lo": lo, "hi": hi}).to_csv(table_path("q1_spline_curve.csv"), index=False)

    candidates = np.arange(15, 46, 0.5)
    log_likelihoods = []
    for threshold in candidates:
        m = pd.DataFrame(
            {
                "seg1": np.minimum(d["vitd"].values, threshold),
                "seg2": np.maximum(d["vitd"].values - threshold, 0),
                "follow_up_years": d["follow_up_years"].values,
                "event_mi_or_stroke": d["event_mi_or_stroke"].values,
            }
        )
        for covar in Q1_COVARS:
            m[covar] = d[covar].values
        try:
            log_likelihoods.append(CoxPHFitter().fit(m, "follow_up_years", "event_mi_or_stroke").log_likelihood_)
        except Exception:
            log_likelihoods.append(np.nan)

    log_likelihoods = np.array(log_likelihoods)
    best = int(np.nanargmax(log_likelihoods))
    threshold_hat = candidates[best]
    inside = candidates[log_likelihoods >= np.nanmax(log_likelihoods) - 1.92]

    segmented = pd.DataFrame(
        {
            "seg1": np.minimum(d["vitd"].values, threshold_hat),
            "seg2": np.maximum(d["vitd"].values - threshold_hat, 0),
            "follow_up_years": d["follow_up_years"].values,
            "event_mi_or_stroke": d["event_mi_or_stroke"].values,
        }
    )
    for covar in Q1_COVARS:
        segmented[covar] = d[covar].values
    cph_segmented = CoxPHFitter().fit(segmented, "follow_up_years", "event_mi_or_stroke")
    cph_segmented.summary.to_csv(table_path("q1_twopiecewise_cox.csv"))

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.plot(candidates, log_likelihoods, color=WCM_NAVY, lw=2)
    ax.axvline(threshold_hat, color=WCM_RED, ls="--", lw=1.8, label=f"Estimate {threshold_hat:.1f} ng/mL")
    ax.axhline(np.nanmax(log_likelihoods) - 1.92, color="grey", ls=":", lw=1.4, label="95% profile likelihood bound")
    ax.axvspan(inside.min(), inside.max(), color=WCM_RED, alpha=0.10)
    ax.axvline(xiao_ng, color="green", ls="-.", lw=1.6, label=f"Xiao 2022 {xiao_ng:.1f} ng/mL")
    ax.set_xlabel("Candidate threshold, ng/mL", fontsize=11)
    ax.set_ylabel("Partial log-likelihood", fontsize=11)
    ax.set_title("Threshold estimation for the L-shaped association", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(figure_path("q1_threshold_profile.png"), dpi=170, bbox_inches="tight")
    plt.show()
    pd.DataFrame({"threshold": candidates, "loglik": log_likelihoods}).to_csv(table_path("q1_threshold_profile.csv"), index=False)

    print_section("Q1 spline and threshold")
    print(f"RCS selected knots: {best_k}")
    print(f"Nonlinearity p: {p_nonlinear:.3e}")
    print(f"Threshold estimate: {threshold_hat:.1f} ng/mL, profile CI {inside.min():.1f}-{inside.max():.1f}")
    print("Saved q1_spline_doseresponse.png, q1_spline_curve.csv, q1_threshold_profile.png, q1_twopiecewise_cox.csv")


## Fit sensitivity models


In [ ]:
run_q1_spline_and_threshold(base)
